# Teacher Selection Spike (Stage T2.1)

This notebook evaluates candidate open-weight Apache 2.0 teacher models on 50 devset items.

## Candidates
1. `Qwen/Qwen3.5-9B` (bf16 baseline on NVIDIA L4)
2. `Qwen/Qwen3.5-27B` (4-bit via bitsandbytes on NVIDIA L4)
3. `Qwen/Qwen3.5-35B-A3B` (MoE: 35B total parameters, 3B active, 4-bit via bitsandbytes on NVIDIA L4)
4. `Qwen/Qwen3.6-27B` (4-bit via bitsandbytes on NVIDIA L4)

## VRAM Sizing for Qwen3.5-35B-A3B (MoE)
- 35B total parameters at 4-bit precision require ~17.5 GB for model weights.
- 3B active parameters require ~2.0 GB for activations and KV cache during generation.
- Total memory requirement (~19.5 GB) fits within the 24 GB VRAM of the NVIDIA L4 GPU.

## Selection Rule
Select the model with the highest `selection_score = pass_rate * mean_reward` (or usefulness) within budget.

In [ ]:
# Step 1: Check GPU environment
!nvidia-smi

In [ ]:
# Step 2: Load Phase 2 Teacher Bundle or Repository Checkout
import os
from pathlib import Path
import shutil
import zipfile

WORK_DIR = Path("/content/Second")
WORK_DIR.mkdir(parents=True, exist_ok=True)

bundle_locations = [
    Path("/content/drive/MyDrive/second/data/phase2_teacher_bundle.zip"),
    Path("/content/phase2_teacher_bundle.zip"),
    Path("data/processed/phase2_teacher_bundle.zip"),
]

bundle_path = None
for loc in bundle_locations:
    if loc.is_file():
        bundle_path = loc
        break

if bundle_path is None:
    if Path("core/python/facts.py").is_file() and Path("eval/devset-1.0.jsonl").is_file():
        print("Detected local repository checkout.")
        WORK_DIR = Path(".").resolve()
    else:
        print("phase2_teacher_bundle.zip not found on Drive or /content.")
        print("Please upload phase2_teacher_bundle.zip using the file picker:")
        try:
            from google.colab import files
            uploaded = files.upload()
            for filename in uploaded.keys():
                if filename.endswith(".zip"):
                    uploaded_path = Path("/content") / filename
                    if not uploaded_path.is_file():
                        uploaded_path = Path(filename).resolve()
                    bundle_path = uploaded_path
                    break
        except Exception as exc:
            print(f"Colab upload notice: {exc}")

if bundle_path and bundle_path.is_file():
    print(f"Extracting bundle from {bundle_path} into {WORK_DIR}...")
    with zipfile.ZipFile(bundle_path, "r") as zf:
        zf.extractall(WORK_DIR)
    print("Bundle extracted successfully.")
    os.chdir(str(WORK_DIR))
elif Path("core/python/facts.py").is_file():
    print("Working in existing checkout.")
else:
    raise FileNotFoundError(
        "Could not find or upload phase2_teacher_bundle.zip. "
        "Run 'python -m training.build_colab_bundle --phase 2' locally and upload the zip."
    )

In [ ]:
# Step 3: Install dependencies
!pip install -q transformers accelerate bitsandbytes chess langdetect

In [ ]:
# Step 4: Run Candidate 1: Qwen/Qwen3.5-9B (bf16)
!python spikes/teacher_selection/run_teacher_spike.py \
    --model Qwen/Qwen3.5-9B \
    --devset eval/devset-1.0.jsonl \
    --count 50 \
    --max-new-tokens 512 \
    --output-dir spikes/teacher_selection/results

In [ ]:
# Step 5: Run Candidate 2: Qwen/Qwen3.5-27B (4-bit)
!python spikes/teacher_selection/run_teacher_spike.py \
    --model Qwen/Qwen3.5-27B \
    --load-in-4bit \
    --devset eval/devset-1.0.jsonl \
    --count 50 \
    --max-new-tokens 512 \
    --output-dir spikes/teacher_selection/results

In [ ]:
# Step 6: Run Candidate 3: Qwen/Qwen3.5-35B-A3B (4-bit)
!python spikes/teacher_selection/run_teacher_spike.py \
    --model Qwen/Qwen3.5-35B-A3B \
    --load-in-4bit \
    --devset eval/devset-1.0.jsonl \
    --count 50 \
    --max-new-tokens 512 \
    --output-dir spikes/teacher_selection/results

In [ ]:
# Step 7: Run Candidate 4: Qwen/Qwen3.6-27B (4-bit)
!python spikes/teacher_selection/run_teacher_spike.py \
    --model Qwen/Qwen3.6-27B \
    --load-in-4bit \
    --devset eval/devset-1.0.jsonl \
    --count 50 \
    --max-new-tokens 512 \
    --output-dir spikes/teacher_selection/results

In [ ]:
# Step 8: Compare Results, Compute Selection Scores, and Print Markdown Table
import json
from pathlib import Path

results_dir = Path("spikes/teacher_selection/results")
summaries = list(results_dir.glob("*_summary.json"))

# Filter out mock summary if real models are present, and omit Qwen3.8
filtered_summaries = [
    s for s in summaries
    if "3_8" not in s.name and "3.8" not in s.name
]
real_summaries = [s for s in filtered_summaries if not s.name.startswith("mock_")]
display_summaries = real_summaries if real_summaries else filtered_summaries

scored_candidates = []
for s_file in display_summaries:
    with open(s_file, "r", encoding="utf-8") as f:
        data = json.load(f)
    pass_rate = data.get("pass_rate", 0.0)
    mean_reward = data.get("mean_reward", 0.0)
    selection_score = pass_rate * mean_reward
    data["selection_score"] = selection_score
    scored_candidates.append(data)

scored_candidates.sort(key=lambda x: x["selection_score"], reverse=True)

print("| Rank | Model | Selection Score | Pass Rate | Mean Reward | In-Band % | Words | Key Point | Time/Item |")
print("|---|---|---|---|---|---|---|---|---|")
for idx, data in enumerate(scored_candidates, 1):
    print(
        f"| #{idx} "
        f"| `{data['model_name']}` "
        f"| **{data['selection_score']:.4f}** "
        f"| {data.get('pass_rate', 0.0):.1%} "
        f"| {data.get('mean_reward', 0.0):.4f} "
        f"| {data.get('in_band_rate', 0.0):.1%} "
        f"| {data.get('mean_word_count', 0.0):.1f} "
        f"| {data.get('mean_key_point', 0.0):.4f} "
        f"| {data.get('seconds_per_item', 0.0):.2f}s |"
    )

if scored_candidates:
    winner = scored_candidates[0]
    print(f"\nWinning candidate: **{winner['model_name']}** (Score: {winner['selection_score']:.4f})")
    print("Action: Record this decision in docs/decisions/teacher_model.md to proceed with Stage T2.3 synthesis.")